# LSTM (PyTorch)

AQI prediction using multi-layer LSTM with sliding window sequences.
Captures temporal dependencies in pollutant time-series data.
Trained independently per city using 10-day lookback windows.
Robust data-driven preprocessing: per-city feature filtering, outlier removal.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [2]:
# Load the multi-city AQI dataset
df = pd.read_csv('../city_day_AQI_pred_dataset.csv')
print(f"Raw dataset: {df.shape[0]} rows, {df.shape[1]} columns")

# Drop rows where AQI (target) is null
df = df.dropna(subset=['AQI'])
print(f"Rows with valid AQI: {len(df)}")

# Drop AQI_Bucket (categorical, derived from AQI)
df = df.drop(columns=['AQI_Bucket'])

df['Date'] = pd.to_datetime(df['Date'])

# All candidate pollutant feature columns
ALL_FEATURE_COLS = ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3',
                    'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']

# City filter: need enough samples for reliable training
MIN_SAMPLES = 300
city_counts = df['City'].value_counts()
valid_cities = city_counts[city_counts >= MIN_SAMPLES].index.tolist()
df = df[df['City'].isin(valid_cities)]

print(f"\nValid cities ({len(valid_cities)}):")
for city in sorted(valid_cities):
    cdf = df[df['City'] == city]
    n = len(cdf)
    miss_pct = cdf[ALL_FEATURE_COLS].isna().mean() * 100
    bad_feats = miss_pct[miss_pct > 50].index.tolist()
    bad_str = f"  (will drop: {', '.join(bad_feats)})" if bad_feats else ""
    print(f"  {city:.<25} {n:>5} samples{bad_str}")

Raw dataset: 29531 rows, 16 columns
Rows with valid AQI: 24850

Valid cities (20):
  Ahmedabad................  1334 samples  (will drop: PM10, NH3)
  Amaravati................   841 samples
  Amritsar.................  1126 samples
  Bengaluru................  1910 samples  (will drop: Xylene)
  Brajrajnagar.............   713 samples  (will drop: Toluene, Xylene)
  Chennai..................  1884 samples  (will drop: PM10, Xylene)
  Coimbatore...............   344 samples  (will drop: Xylene)
  Delhi....................  1999 samples
  Gurugram.................  1453 samples  (will drop: NH3, Xylene)
  Guwahati.................   495 samples  (will drop: Toluene, Xylene)
  Hyderabad................  1880 samples
  Jaipur...................  1094 samples  (will drop: Xylene)
  Jorapokhar...............   771 samples  (will drop: PM2.5, NOx, Benzene, Toluene, Xylene)
  Kolkata..................   754 samples
  Lucknow..................  1893 samples  (will drop: PM10, Xylene)
  Mumbai.

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
df.boxplot(column='AQI', by='City', ax=axes[0], rot=45)
axes[0].set_title('AQI Distribution by City')
axes[0].set_xlabel(''); axes[0].set_ylabel('AQI')
axes[0].get_figure().suptitle('')
miss = df[ALL_FEATURE_COLS].isna().sum().sort_values(ascending=True)
axes[1].barh(miss.index, miss.values, color='coral')
axes[1].set_title('Missing Values per Feature (all cities)')
axes[1].set_xlabel('Count')
plt.tight_layout(); plt.show()

plt.figure(figsize=(10, 8))
corr = df[ALL_FEATURE_COLS + ['AQI']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Feature-AQI Correlation Matrix')
plt.tight_layout(); plt.show()

In [4]:
MISSING_THRESHOLD = 0.5
MIN_FEATURES = 3
VARIANCE_THRESHOLD = 0.01
SEQ_LENGTH = 10

def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len])
    return np.array(Xs), np.array(ys)

def preprocess_city_lstm(df, city, all_feature_cols, seq_len, test_ratio=0.2):
    city_df = df[df['City'] == city].copy()
    city_df = city_df.sort_values('Date').reset_index(drop=True)

    missing_pct = city_df[all_feature_cols].isna().mean()
    usable_features = missing_pct[missing_pct < MISSING_THRESHOLD].index.tolist()
    dropped = sorted(set(all_feature_cols) - set(usable_features))
    if dropped:
        print(f"  Dropped {len(dropped)} sparse features: {dropped}")
    if len(usable_features) < MIN_FEATURES:
        print(f"  [SKIP] {city}: only {len(usable_features)} usable features")
        return None

    city_df[usable_features] = city_df[usable_features].interpolate(
        method='linear', limit_direction='both'
    )
    for col in usable_features:
        if city_df[col].isna().any():
            city_df[col] = city_df[col].fillna(city_df[col].median())
    city_df = city_df.dropna(subset=usable_features + ['AQI'])

    Q1, Q3 = city_df['AQI'].quantile(0.25), city_df['AQI'].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_before = len(city_df)
    city_df = city_df[(city_df['AQI'] >= lower) & (city_df['AQI'] <= upper)]
    n_outliers = n_before - len(city_df)
    if n_outliers > 0:
        print(f"  Removed {n_outliers} AQI outliers (IQR: {lower:.0f}-{upper:.0f})")

    variances = city_df[usable_features].var()
    low_var = variances[variances < VARIANCE_THRESHOLD].index.tolist()
    if low_var:
        usable_features = [f for f in usable_features if f not in low_var]
        print(f"  Dropped {len(low_var)} low-variance features: {low_var}")

    if len(usable_features) < MIN_FEATURES or len(city_df) < seq_len + 50:
        print(f"  [SKIP] {city}: insufficient data after cleaning")
        return None

    X_raw = city_df[usable_features].values.astype(np.float32)
    y_raw = city_df['AQI'].values.astype(np.float32)

    sc_x = StandardScaler()
    sc_y = StandardScaler()
    X_scaled = sc_x.fit_transform(X_raw).astype(np.float32)
    y_scaled = sc_y.fit_transform(y_raw.reshape(-1, 1)).flatten().astype(np.float32)

    X_seq, y_seq = create_sequences(X_scaled, y_scaled, seq_len)
    _, y_orig_seq = create_sequences(X_raw, y_raw, seq_len)

    split = int((1 - test_ratio) * len(X_seq))
    X_train, X_test = X_seq[:split], X_seq[split:]
    y_train, y_test = y_seq[:split], y_seq[split:]
    y_test_orig = y_orig_seq[split:]

    print(f"  Features used ({len(usable_features)}): {usable_features}")
    print(f"  Sequences: {len(X_train)} train / {len(X_test)} test")

    return {
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'y_test_orig': y_test_orig,
        'sc_x': sc_x, 'sc_y': sc_y,
        'n_train': len(X_train), 'n_test': len(X_test),
        'n_features': len(usable_features),
        'features': usable_features
    }

In [5]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size, hidden_size=hidden_size,
            num_layers=num_layers, batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.fc(lstm_out[:, -1, :])

In [6]:
def train_lstm(model, X_train, y_train, X_val, y_val,
               num_epochs=500, lr=0.001, batch_size=32, patience=30):
    model = model.to(device)
    train_ds = TensorDataset(
        torch.FloatTensor(X_train).to(device),
        torch.FloatTensor(y_train).to(device)
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=10, factor=0.5, min_lr=1e-6
    )
    X_val_t = torch.FloatTensor(X_val).to(device)
    y_val_t = torch.FloatTensor(y_val).to(device)

    best_val = float('inf')
    best_state = None
    no_improve = 0
    history = {'train': [], 'val': []}

    for epoch in range(num_epochs):
        model.train()
        ep_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            out = model(xb).squeeze(-1)
            loss = criterion(out, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            ep_loss += loss.item()

        avg_train = ep_loss / len(train_loader)
        history['train'].append(avg_train)

        model.eval()
        with torch.no_grad():
            v_pred = model(X_val_t).squeeze(-1)
            v_loss = criterion(v_pred, y_val_t).item()
        history['val'].append(v_loss)
        scheduler.step(v_loss)

        if v_loss < best_val:
            best_val = v_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience and epoch > 50:
            break

    if best_state:
        model.load_state_dict(best_state)
    return model, history

In [7]:
def evaluate_lstm(model, X_test, y_test_orig, sc_y):
    model.eval()
    with torch.no_grad():
        preds_s = model(torch.FloatTensor(X_test).to(device)).squeeze(-1).cpu().numpy()
    preds = sc_y.inverse_transform(preds_s.reshape(-1, 1)).flatten()
    return {
        'predictions': preds, 'actuals': y_test_orig,
        'r2': r2_score(y_test_orig, preds),
        'mse': mean_squared_error(y_test_orig, preds),
        'rmse': np.sqrt(mean_squared_error(y_test_orig, preds)),
        'mae': mean_absolute_error(y_test_orig, preds),
        'mape': np.mean(np.abs((y_test_orig - preds) / np.where(y_test_orig == 0, 1, y_test_orig))) * 100
    }

In [8]:
def plot_results(all_results, model_name):
    n = len(all_results)
    fig, axes = plt.subplots(n, 1, figsize=(14, 4 * n))
    if n == 1: axes = [axes]
    for ax, (city, res) in zip(axes, all_results.items()):
        ax.plot(res['actuals'], label='Actual AQI', color='steelblue', alpha=0.8, lw=1.2)
        ax.plot(res['predictions'], label='Predicted AQI', color='tomato', alpha=0.8, lw=1.2)
        ax.set_title(f'{city}  |  R² = {res["r2"]:.4f}  |  RMSE = {res["rmse"]:.2f}  |  MAE = {res["mae"]:.2f}',
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('Test Sample'); ax.set_ylabel('AQI')
        ax.legend(loc='upper right'); ax.grid(True, alpha=0.3)
    plt.suptitle(f'{model_name} — Per-City AQI Predictions', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout(); plt.show()

def plot_training_curves(all_histories, model_name):
    n = len(all_histories)
    cols = min(3, n); rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    axes = np.array(axes).flatten() if n > 1 else [axes]
    for ax, (city, hist) in zip(axes, all_histories.items()):
        ax.plot(hist['train'], label='Train', alpha=0.8)
        ax.plot(hist['val'], label='Val', alpha=0.8)
        ax.set_title(city, fontsize=10); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    for ax in axes[n:]: ax.set_visible(False)
    plt.suptitle(f'{model_name} — Training Curves', fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

In [9]:
all_results = {}
all_histories = {}

for city in sorted(valid_cities):
    print(f"\n{'='*60}")
    print(f"  LSTM — {city}")
    print(f"{'='*60}")

    data = preprocess_city_lstm(df, city, ALL_FEATURE_COLS, SEQ_LENGTH)
    if data is None:
        continue

    input_size = data['n_features']
    model = LSTMModel(input_size=input_size, hidden_size=64, num_layers=2, dropout=0.2)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {n_params:,}")

    model, history = train_lstm(
        model, data['X_train'], data['y_train'],
        data['X_test'], data['y_test'],
        num_epochs=500, lr=0.001, batch_size=32, patience=30
    )

    results = evaluate_lstm(model, data['X_test'], data['y_test_orig'], data['sc_y'])
    results['n_train'] = data['n_train']
    results['n_test'] = data['n_test']
    results['n_features'] = data['n_features']
    all_results[city] = results
    all_histories[city] = history

    print(f"  >> R²={results['r2']:.4f} | RMSE={results['rmse']:.2f} | MAE={results['mae']:.2f}")

print(f"\nDone! Trained LSTM on {len(all_results)} cities.")


  LSTM — Ahmedabad
  Dropped 2 sparse features: ['NH3', 'PM10']
  Removed 92 AQI outliers (IQR: -237-1010)
  Features used (10): ['PM2.5', 'NO', 'NO2', 'NOx', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
  Sequences: 985 train / 247 test
  Parameters: 56,961


  >> R²=0.6589 | RMSE=109.86 | MAE=74.11

  LSTM — Amaravati
  Removed 52 AQI outliers (IQR: -35-205)
  Features used (12): ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
  Sequences: 623 train / 156 test
  Parameters: 57,473


  >> R²=0.2664 | RMSE=14.21 | MAE=9.80

  LSTM — Amritsar
  Removed 95 AQI outliers (IQR: -23-232)
  Features used (12): ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
  Sequences: 816 train / 205 test
  Parameters: 57,473


  >> R²=0.6131 | RMSE=24.05 | MAE=16.97

  LSTM — Bengaluru
  Dropped 1 sparse features: ['Xylene']
  Removed 71 AQI outliers (IQR: 1-177)
  Features used (11): ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene']
  Sequences: 1463 train / 366 test
  Parameters: 57,217


  >> R²=0.7542 | RMSE=9.87 | MAE=7.24

  LSTM — Brajrajnagar
  Dropped 2 sparse features: ['Toluene', 'Xylene']
  Removed 18 AQI outliers (IQR: -24-316)
  Features used (10): ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene']
  Sequences: 548 train / 137 test
  Parameters: 56,961


...

  >> R²=0.9037 | RMSE=13.42 | MAE=9.59

  LSTM — Patna
  Dropped 2 sparse features: ['NH3', 'PM10']
  Features used (10): ['PM2.5', 'NO', 'NO2', 'NOx', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
  Sequences: 1159 train / 290 test
  Parameters: 56,961


  >> R²=0.8987 | RMSE=34.20 | MAE=25.77

  LSTM — Talcher
  Dropped 2 sparse features: ['Toluene', 'Xylene']
  Removed 9 AQI outliers (IQR: -101-441)
  Features used (10): ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene']
  Sequences: 543 train / 136 test
  Parameters: 56,961


  >> R²=0.7901 | RMSE=34.25 | MAE=23.89

  LSTM — Thiruvananthapuram
  Dropped 3 sparse features: ['Benzene', 'Toluene', 'Xylene']
  Removed 43 AQI outliers (IQR: 9-137)
  Features used (9): ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3']
  Sequences: 799 train / 200 test
  Parameters: 56,705


  >> R²=0.6763 | RMSE=10.63 | MAE=8.49

  LSTM — Visakhapatnam
  Removed 87 AQI outliers (IQR: 6-210)
  Features used (12): ['PM2.5', 'PM10', 'NO', 'NO2', 'NOx', 'NH3', 'CO', 'SO2', 'O3', 'Benzene', 'Toluene', 'Xylene']
  Sequences: 859 train / 215 test
  Parameters: 57,473


  >> R²=0.7084 | RMSE=18.04 | MAE=13.54

Done! Trained LSTM on 20 cities.


In [10]:
print("\n" + "=" * 90)
print(f"  PERFORMANCE SUMMARY")
print("=" * 90)
print(f"  {'City':<20} {'R²':>8} {'RMSE':>10} {'MAE':>10} {'MSE':>12} {'Features':>10} {'Split':>12} {'Accuracy%':>10}")
print("-" * 90)
for city in sorted(all_results.keys()):
    r = all_results[city]
    nf = r.get('n_features', '?')
    flag = " ⚠" if r['r2'] < 0.5 else ""
    print(f"  {city:<20} {r['r2']:>8.4f} {r['rmse']:>10.2f} {r['mae']:>10.2f} {r['mse']:>12.2f} {nf:>10} {r['n_train']}+{r['n_test']}  {max(0, 100 - r.get('mape', 0)):>8.1f}%{flag}")
print("-" * 90)
avg_r2 = np.mean([r['r2'] for r in all_results.values()])
avg_rmse = np.mean([r['rmse'] for r in all_results.values()])
avg_mae = np.mean([r['mae'] for r in all_results.values()])
avg_acc = np.mean([max(0, 100 - r.get('mape', 0)) for r in all_results.values()])
print(f"  {'AVERAGE':<20} {avg_r2:>8.4f} {avg_rmse:>10.2f} {avg_mae:>10.2f}{'':>12} {'':>10} {avg_acc:>8.1f}%")
print("=" * 90)
if any(r['r2'] < 0.5 for r in all_results.values()):
    print("  ⚠ = R² < 0.5 (weak model — likely due to data quality or distribution shift)")


  PERFORMANCE SUMMARY
  City                       R²       RMSE        MAE          MSE   Features        Split  Accuracy%
------------------------------------------------------------------------------------------
  Ahmedabad              0.6589     109.86      74.11     12069.60         10 985+247      66.7%
  Amaravati              0.2664      14.21       9.80       201.90         12 623+156      80.9% ⚠
  Amritsar               0.6131      24.05      16.97       578.22         12 816+205      81.3%
  Bengaluru              0.7542       9.87       7.24        97.36         11 1463+366      90.7%
  Brajrajnagar           0.2727      43.14      32.37      1860.75         10 548+137      73.6% ⚠
  Chennai                0.2972      21.15      15.89       447.43         10 1444+362      80.5% ⚠
  Coimbatore            -0.8867      27.54      24.41       758.31          9 260+66      44.7% ⚠
  Delhi                  0.9205      31.50      23.36       992.25         12 1586+397      86.4

In [11]:
plot_results(all_results, 'LSTM')

In [12]:
plot_training_curves(all_histories, 'LSTM')